In [1]:
import h5py
import numpy as np
import torch
import torch.nn as nn

import torch.functional as F

from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset, Dataset
import torchmetrics

from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
from sklearn.metrics import classification_report

# Подготовка датасета

In [2]:
path = '/kaggle/input/datasets/alexandra841/ordinary-spec/features_ordinary_spec.h5'

In [3]:
with h5py.File(path, 'r') as f:
    spec = f['spec'][:]
    label = f['class_mood_int'][:]

In [6]:
class SpecDataset(Dataset):

    def __init__(self, spec, label):
        self.spec = spec
        self.label = label

    def __len__(self):
        return len(self.label)

    def __getitem__(self, idx):

        spec = (self.spec)[idx]
        label = (self.label)[idx]
        
        spec = torch.tensor(spec)
        label = torch.tensor(label)

       # spec = (spec - spec.mean()) / (spec.std() + 1e-6)

        return spec, label

In [7]:
dataset = SpecDataset(spec, label)

In [8]:
labels = dataset[:][1]

In [9]:
torch.unique(labels, return_counts=True)

(tensor([0, 1, 2, 3, 4, 5], dtype=torch.int8),
 tensor([4326, 1987, 5126, 2452,  793,  919]))

In [10]:
train_val_idx, test_idx = train_test_split(
    np.arange(len(dataset)), test_size=0.2, shuffle=True, random_state=42, stratify=labels
)

trainvalset = torch.utils.data.Subset(dataset, train_val_idx)
testset = torch.utils.data.Subset(dataset, test_idx)

In [11]:
labels_train = labels[train_val_idx]

In [12]:
torch.unique(labels_train, return_counts=True)

(tensor([0, 1, 2, 3, 4, 5], dtype=torch.int8),
 tensor([3461, 1590, 4101, 1961,  634,  735]))

In [13]:
train_idx, val_idx = train_test_split(
    np.arange(len(trainvalset)), test_size=0.2, shuffle=True, random_state=42, stratify=labels_train
)

trainset = torch.utils.data.Subset(trainvalset, train_idx)
valset = torch.utils.data.Subset(trainvalset, val_idx)

In [14]:
train_loader = DataLoader(trainset, batch_size=16, shuffle=True, num_workers=4, drop_last=True)
val_loader = DataLoader(valset, batch_size=16, shuffle=False, num_workers=4)
test_loader = DataLoader(testset, batch_size=16, shuffle=False,  num_workers=4)

In [15]:
n_classes = 6
in_channels = 1

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# Модель 1

## Архитектура

In [29]:
class CNNModel(nn.Module):
    
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, 8, kernel_size=5, padding='same')
        self.bn1 = nn.BatchNorm2d(8)
        self.relu1 = nn.LeakyReLU()
        self.max_pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(8, 16, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm2d(16)
        self.relu2 = nn.LeakyReLU()
        self.max_pool2 = nn.MaxPool2d(2)


        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(16, n_classes)

    def forward(self, x):
        x = self.max_pool1(self.relu1(self.bn1(self.conv1(x))))
        x = self.max_pool2(self.relu2(self.bn2(self.conv2(x))))

        x = self.gap(x)
        x = torch.flatten(x, 1)

        x = self.fc(x)
        return x

In [30]:
model = CNNModel()

In [31]:
model = model.to(device)

In [32]:
class_weights = compute_class_weight(
        'balanced',
        classes=np.arange(n_classes),
        y=labels_train.numpy()
    )

In [33]:
class_weights_tensor = torch.tensor(
    class_weights, dtype=torch.float32
).to(device)

In [34]:
class_weights_tensor

tensor([0.6011, 1.3084, 0.5073, 1.0609, 3.2813, 2.8304], device='cuda:0')

## Обучение

In [35]:
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

In [36]:
def train_epoch(model, optimizer, train_loader, loss_fn, n_classes):

    loss_log = []
    acc_log = []
    f1_log = []
    
    model.train()

    acc_calc = torchmetrics.classification.Accuracy(
        num_classes=n_classes, task="multiclass"
    ).to(device)

    f1_calc = torchmetrics.classification.F1Score(
        num_classes=n_classes, task="multiclass", average="macro"
    ).to(device)

    for spec, label in train_loader:
        spec = spec.to(device).float()
        label = label.to(device).long()

        optimizer.zero_grad()
        output_log = model(spec)
        output = torch.argmax(output_log, dim=1)

        loss = loss_fn(output_log, label)
        loss_log.append(loss.item())
        
        #acc = acc_calc(output, label)
        acc_calc.update(output, label)

        #f1 = f1_calc(output, label)
        f1_calc.update(output, label)

        loss.backward()
        optimizer.step()

    acc = acc_calc.compute().item()
    f1 = f1_calc.compute().item()

    acc_calc.reset()
    f1_calc.reset()
        
    return np.mean(loss_log), acc, f1

def test(model, loader, loss_fn, n_classes):
    
    loss_log = []
    preds_log = []
    labels_log = []
    
    model.eval()
    
    acc_calc = torchmetrics.classification.Accuracy(
        num_classes=n_classes, task="multiclass"
    ).to(device)

    f1_calc = torchmetrics.classification.F1Score(
        num_classes=n_classes, task="multiclass", average="macro"
    ).to(device)

    with torch.no_grad():
        for spec, label in loader:
            spec = spec.to(device).float()
            label = label.to(device).long()
            labels_log.extend(list(label.cpu().numpy()))
            output_log = model(spec)
            output = torch.argmax(output_log, dim=1)
            preds_log.extend(list(output.cpu().numpy()))
    
            loss = loss_fn(output_log, label)
            loss_log.append(loss.item())

            #acc = acc_calc(output, label)
            acc_calc.update(output, label)
    
            #f1 = f1_calc(output, label)
            f1_calc.update(output, label)

    acc = acc_calc.compute().item()
    f1 = f1_calc.compute().item()

    acc_calc.reset()
    f1_calc.reset()
        
    return np.mean(loss_log), acc, f1, preds_log, labels_log

def train(model, optimizer, n_epoch, train_loader, val_loader, loss_fn, scheduler=None, n_classes=6):
    best_f1 = 0
    
    train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = [], [], [], [], [], []

    for epoch in tqdm(range(n_epoch)):
        train_loss, train_acc, train_f1 = train_epoch(model, optimizer, train_loader, loss_fn, n_classes)
        val_loss, val_acc, val_f1, preds_log, labels_log = test(model, val_loader, loss_fn, n_classes)
        
        train_loss_log.append(train_loss)
        train_acc_log.append(train_acc)
        train_f1_log.append(train_f1)

        val_loss_log.append(val_loss)
        val_acc_log.append(val_acc)
        val_f1_log.append(val_f1)

        print(f"Epoch {epoch}")
        print(f" train loss: {train_loss}, train acc: {train_acc}, train f1: {train_f1}")
        print(f" val loss: {val_loss}, val acc: {val_acc}, val f1: {val_f1}\n")
        print(classification_report(np.array(labels_log), np.array(preds_log)))

        if scheduler is not None:
            if isinstance(scheduler,torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_f1)
            else:
                scheduler.step()


        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), "best_model.pt")

    return train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log

In [37]:
optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0.01, lr=2e-4)

In [38]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.2, patience=1
)

In [39]:
train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = train(
    model,
    optimizer,
    20,
    train_loader, 
    val_loader, 
    loss_fn,
    scheduler
)

  0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
  5%|▌     

Epoch 0
 train loss: 1.770477329691251, train acc: 0.27984777092933655, train f1: 0.1510361284017563
 val loss: 1.7495531823225081, val acc: 0.35162195563316345, val f1: 0.1855437308549881

              precision    recall  f1-score   support

           0       0.36      0.77      0.49       692
           1       0.00      0.00      0.00       318
           2       0.43      0.23      0.30       821
           3       0.28      0.39      0.33       392
           4       0.00      0.00      0.00       127
           5       0.00      0.00      0.00       147

    accuracy                           0.35      2497
   macro avg       0.18      0.23      0.19      2497
weighted avg       0.28      0.35      0.28      2497



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
 10%|█         | 2/20 [00:09<01:24,  4.68s/it]

Epoch 1
 train loss: 1.7389097169805796, train acc: 0.33974358439445496, train f1: 0.21075475215911865
 val loss: 1.7319192051128218, val acc: 0.3315979242324829, val f1: 0.22797685861587524

              precision    recall  f1-score   support

           0       0.35      0.59      0.44       692
           1       0.16      0.14      0.15       318
           2       0.47      0.27      0.34       821
           3       0.31      0.36      0.33       392
           4       0.00      0.00      0.00       127
           5       0.09      0.10      0.10       147

    accuracy                           0.33      2497
   macro avg       0.23      0.24      0.23      2497
weighted avg       0.33      0.33      0.31      2497



 15%|█▌        | 3/20 [00:13<01:18,  4.61s/it]

...

Epoch 17
 train loss: 1.7205618593173149, train acc: 0.33383414149284363, train f1: 0.2424432337284088
 val loss: 1.7182969948288742, val acc: 0.32959550619125366, val f1: 0.23012349009513855

              precision    recall  f1-score   support

           0       0.36      0.60      0.45       692
           1       0.17      0.17      0.17       318
           2       0.48      0.24      0.32       821
           3       0.31      0.36      0.33       392
           4       0.00      0.00      0.00       127
           5       0.09      0.12      0.10       147

    accuracy                           0.33      2497
   macro avg       0.24      0.25      0.23      2497
weighted avg       0.34      0.33      0.31      2497



 95%|█████████▌| 19/20 [01:28<00:04,  4.68s/it]

Epoch 18
 train loss: 1.719342586130668, train acc: 0.3335336446762085, train f1: 0.24154099822044373
 val loss: 1.7169871178402263, val acc: 0.322787344455719, val f1: 0.23290038108825684

              precision    recall  f1-score   support

           0       0.36      0.53      0.43       692
           1       0.15      0.16      0.16       318
           2       0.48      0.25      0.33       821
           3       0.31      0.40      0.35       392
           4       0.00      0.00      0.00       127
           5       0.11      0.16      0.13       147

    accuracy                           0.32      2497
   macro avg       0.24      0.25      0.23      2497
weighted avg       0.33      0.32      0.31      2497



100%|██████████| 20/20 [01:32<00:00,  4.64s/it]

Epoch 19
 train loss: 1.7190907929952328, train acc: 0.3322315812110901, train f1: 0.24028442800045013
 val loss: 1.7179832595169164, val acc: 0.3195835053920746, val f1: 0.2278052717447281

              precision    recall  f1-score   support

           0       0.36      0.55      0.44       692
           1       0.16      0.18      0.17       318
           2       0.48      0.23      0.31       821
           3       0.30      0.40      0.34       392
           4       0.00      0.00      0.00       127
           5       0.10      0.12      0.11       147

    accuracy                           0.32      2497
   macro avg       0.23      0.25      0.23      2497
weighted avg       0.33      0.32      0.30      2497



# Модель 2

In [40]:
class CNNModel2(nn.Module):
    
    def __init__(self):
        super().__init__()   

        self.block1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm2d(16),
            nn.SiLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.MaxPool2d(2)
        )

        self.gap = nn.AdaptiveAvgPool2d((4, 4))

        self.fc = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        x = self.gap(x)
        x = torch.flatten(x, 1)

        x = self.fc(x)
        return x

In [45]:
model2 = CNNModel2()
model2 = model2.float()
model2 = model2.to(device)

In [46]:
optimizer = torch.optim.AdamW(model2.parameters(), lr=1e-4, weight_decay=1e-4)

In [47]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3
)

In [48]:
train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = train(
    model2,
    optimizer,
    20,
    train_loader, 
    val_loader, 
    loss_fn,
    scheduler
)

  5%|▌         | 1/20 [00:06<02:03,  6.52s/it]

Epoch 0
 train loss: 1.7472239839725006, train acc: 0.30418670177459717, train f1: 0.22141703963279724
 val loss: 1.722651701823921, val acc: 0.3488185703754425, val f1: 0.21576014161109924

              precision    recall  f1-score   support

           0       0.37      0.71      0.48       692
           1       0.08      0.01      0.01       318
           2       0.45      0.33      0.38       821
           3       0.33      0.22      0.27       392
           4       0.07      0.07      0.07       127
           5       0.08      0.08      0.08       147

    accuracy                           0.35      2497
   macro avg       0.23      0.24      0.22      2497
weighted avg       0.32      0.35      0.31      2497



 10%|█         | 2/20 [00:13<01:57,  6.52s/it]

Epoch 1
 train loss: 1.7061473910625164, train acc: 0.31780847907066345, train f1: 0.2508459985256195
 val loss: 1.69805575631986, val acc: 0.2943532168865204, val f1: 0.2346067726612091

              precision    recall  f1-score   support

           0       0.43      0.44      0.43       692
           1       0.14      0.08      0.11       318
           2       0.48      0.19      0.27       821
           3       0.27      0.52      0.35       392
           4       0.08      0.19      0.11       127
           5       0.11      0.17      0.13       147

    accuracy                           0.29      2497
   macro avg       0.25      0.26      0.23      2497
weighted avg       0.35      0.29      0.29      2497



 15%|█▌        | 3/20 [00:19<01:50,  6.50s/it]

...

Epoch 17
 train loss: 1.5192651838446274, train acc: 0.3835136294364929, train f1: 0.34090012311935425
 val loss: 1.663275989757222, val acc: 0.3331998288631439, val f1: 0.2816590964794159

              precision    recall  f1-score   support

           0       0.51      0.36      0.42       692
           1       0.25      0.09      0.13       318
           2       0.57      0.39      0.47       821
           3       0.36      0.34      0.35       392
           4       0.10      0.44      0.16       127
           5       0.11      0.29      0.16       147

    accuracy                           0.33      2497
   macro avg       0.32      0.32      0.28      2497
weighted avg       0.43      0.33      0.36      2497



 95%|█████████▌| 19/20 [02:02<00:06,  6.42s/it]

Epoch 18
 train loss: 1.5127626413909288, train acc: 0.38721954822540283, train f1: 0.3481979966163635
 val loss: 1.6813406967053748, val acc: 0.3432118594646454, val f1: 0.2908346354961395

              precision    recall  f1-score   support

           0       0.46      0.46      0.46       692
           1       0.18      0.21      0.20       318
           2       0.58      0.34      0.43       821
           3       0.36      0.29      0.32       392
           4       0.12      0.27      0.17       127
           5       0.11      0.29      0.16       147

    accuracy                           0.34      2497
   macro avg       0.30      0.31      0.29      2497
weighted avg       0.41      0.34      0.36      2497



100%|██████████| 20/20 [02:09<00:00,  6.45s/it]

Epoch 19
 train loss: 1.494592441389194, train acc: 0.39403045177459717, train f1: 0.35633426904678345
 val loss: 1.663134498201358, val acc: 0.35883060097694397, val f1: 0.2966883182525635

              precision    recall  f1-score   support

           0       0.49      0.38      0.43       692
           1       0.21      0.15      0.18       318
           2       0.55      0.42      0.48       821
           3       0.32      0.43      0.37       392
           4       0.13      0.27      0.18       127
           5       0.12      0.24      0.16       147

    accuracy                           0.36      2497
   macro avg       0.30      0.32      0.30      2497
weighted avg       0.41      0.36      0.37      2497



# Модель 3

In [49]:
class CNNModel3(nn.Module):
    
    def __init__(self):
        super().__init__()   

        self.block1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm2d(16),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.05)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.MaxPool2d(2),
            #nn.Dropout2d(0.1)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.05)
        )

        self.gap = nn.AdaptiveAvgPool2d((8, 8))

        self.fc = nn.Sequential(
            nn.Linear(64 * 8 * 8, 256),
            nn.SiLU(),
            #nn.Dropout(0.1),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        x = self.gap(x)
        x = torch.flatten(x, 1)

        x = self.fc(x)
        return x

In [50]:
model3 = CNNModel3()
model3 = model3.to(device)

In [51]:
loss_fn3 = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    label_smoothing=0.1
)

In [52]:
optimizer = torch.optim.AdamW(model3.parameters(), lr=1e-4, weight_decay=1e-4)

In [53]:
train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = train(
    model3,
    optimizer,
    20,
    train_loader, 
    val_loader, 
    loss_fn
)

  5%|▌         | 1/20 [00:06<02:08,  6.74s/it]

Epoch 0
 train loss: 1.746179750141425, train acc: 0.28896233439445496, train f1: 0.2280861735343933
 val loss: 1.713501730542274, val acc: 0.2943532168865204, val f1: 0.22705954313278198

              precision    recall  f1-score   support

           0       0.42      0.31      0.36       692
           1       0.19      0.02      0.03       318
           2       0.44      0.34      0.39       821
           3       0.29      0.44      0.35       392
           4       0.08      0.17      0.11       127
           5       0.08      0.27      0.13       147

    accuracy                           0.29      2497
   macro avg       0.25      0.26      0.23      2497
weighted avg       0.34      0.29      0.30      2497



 10%|█         | 2/20 [00:13<02:01,  6.72s/it]

Epoch 1
 train loss: 1.698438859711855, train acc: 0.3294270932674408, train f1: 0.2637064754962921
 val loss: 1.6914382557960073, val acc: 0.33440127968788147, val f1: 0.25167426466941833

              precision    recall  f1-score   support

           0       0.42      0.37      0.39       692
           1       0.16      0.32      0.21       318
           2       0.46      0.44      0.45       821
           3       0.30      0.24      0.27       392
           4       0.12      0.03      0.05       127
           5       0.14      0.13      0.13       147

    accuracy                           0.33      2497
   macro avg       0.27      0.25      0.25      2497
weighted avg       0.35      0.33      0.34      2497



 15%|█▌        | 3/20 [00:20<01:54,  6.71s/it]

...

Epoch 17
 train loss: 0.9047904302103397, train acc: 0.625901460647583, train f1: 0.6130440831184387
 val loss: 2.0999308960263137, val acc: 0.37845414876937866, val f1: 0.3110145926475525

              precision    recall  f1-score   support

           0       0.47      0.46      0.47       692
           1       0.25      0.30      0.27       318
           2       0.55      0.38      0.45       821
           3       0.31      0.43      0.36       392
           4       0.18      0.13      0.15       127
           5       0.13      0.22      0.17       147

    accuracy                           0.38      2497
   macro avg       0.32      0.32      0.31      2497
weighted avg       0.41      0.38      0.39      2497



 95%|█████████▌| 19/20 [02:06<00:06,  6.66s/it]

Epoch 18
 train loss: 0.866763560531231, train acc: 0.6438301205635071, train f1: 0.6292080879211426
 val loss: 2.100536817957641, val acc: 0.3560272455215454, val f1: 0.29931771755218506

              precision    recall  f1-score   support

           0       0.50      0.36      0.42       692
           1       0.25      0.30      0.27       318
           2       0.52      0.41      0.46       821
           3       0.29      0.39      0.33       392
           4       0.11      0.29      0.16       127
           5       0.17      0.14      0.15       147

    accuracy                           0.36      2497
   macro avg       0.31      0.31      0.30      2497
weighted avg       0.40      0.36      0.37      2497



100%|██████████| 20/20 [02:13<00:00,  6.67s/it]

Epoch 19
 train loss: 0.8184727781380599, train acc: 0.6671674847602844, train f1: 0.6578975915908813
 val loss: 2.0875513010723576, val acc: 0.33039647340774536, val f1: 0.28831592202186584

              precision    recall  f1-score   support

           0       0.46      0.32      0.37       692
           1       0.20      0.50      0.28       318
           2       0.56      0.33      0.42       821
           3       0.33      0.32      0.32       392
           4       0.14      0.22      0.17       127
           5       0.17      0.16      0.17       147

    accuracy                           0.33      2497
   macro avg       0.31      0.31      0.29      2497
weighted avg       0.40      0.33      0.34      2497



# Модель 4

In [59]:
class CNNModel4(nn.Module):
    
    def __init__(self):
        super().__init__()   

        self.block1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2),
            nn.BatchNorm2d(16),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.02)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.MaxPool2d(2),
            #nn.Dropout2d(0.1)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.02)
        )

        self.gap = nn.AdaptiveAvgPool2d((6, 6))

        self.fc = nn.Sequential(
            nn.Linear(64 * 6 * 6, 256),
            nn.SiLU(),
            #nn.Dropout(0.1),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        x = self.gap(x)
        x = torch.flatten(x, 1)

        x = self.fc(x)
        return x

In [60]:
model4 = CNNModel4()
model4 = model4.to(device)

In [61]:
optimizer = torch.optim.AdamW(
    model4.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

In [62]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=25
)

In [63]:
train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = train(
    model4,
    optimizer,
    25,
    train_loader, 
    val_loader, 
    loss_fn,
    scheduler
)

  4%|▍         | 1/25 [00:06<02:37,  6.57s/it]

Epoch 0
 train loss: 1.741556095962341, train acc: 0.28745993971824646, train f1: 0.22364835441112518
 val loss: 1.7000127802988527, val acc: 0.3432118594646454, val f1: 0.24542126059532166

              precision    recall  f1-score   support

           0       0.41      0.50      0.45       692
           1       0.18      0.05      0.08       318
           2       0.44      0.44      0.44       821
           3       0.31      0.22      0.26       392
           4       0.09      0.17      0.11       127
           5       0.11      0.15      0.13       147

    accuracy                           0.34      2497
   macro avg       0.26      0.26      0.25      2497
weighted avg       0.34      0.34      0.33      2497



  8%|▊         | 2/25 [00:13<02:30,  6.56s/it]

Epoch 1
 train loss: 1.6966449339420369, train acc: 0.3192107379436493, train f1: 0.2571428418159485
 val loss: 1.7012563936269967, val acc: 0.3163796663284302, val f1: 0.22169411182403564

              precision    recall  f1-score   support

           0       0.41      0.56      0.47       692
           1       0.00      0.00      0.00       318
           2       0.46      0.32      0.38       821
           3       0.33      0.18      0.23       392
           4       0.09      0.12      0.10       127
           5       0.09      0.37      0.15       147

    accuracy                           0.32      2497
   macro avg       0.23      0.26      0.22      2497
weighted avg       0.33      0.32      0.31      2497



 12%|█▏        | 3/25 [00:19<02:24,  6.56s/it]

...

Epoch 22
 train loss: 1.3042474201856515, train acc: 0.4704527258872986, train f1: 0.44002223014831543
 val loss: 1.7050325843938596, val acc: 0.34921905398368835, val f1: 0.2985804080963135

              precision    recall  f1-score   support

           0       0.50      0.37      0.43       692
           1       0.25      0.23      0.24       318
           2       0.56      0.37      0.45       821
           3       0.31      0.42      0.36       392
           4       0.13      0.32      0.18       127
           5       0.11      0.21      0.15       147

    accuracy                           0.35      2497
   macro avg       0.31      0.32      0.30      2497
weighted avg       0.41      0.35      0.37      2497



 96%|█████████▌| 24/25 [02:38<00:06,  6.62s/it]

Epoch 23
 train loss: 1.2984075906376045, train acc: 0.4771634638309479, train f1: 0.44544246792793274
 val loss: 1.6999186956958405, val acc: 0.34681618213653564, val f1: 0.29877805709838867

              precision    recall  f1-score   support

           0       0.50      0.36      0.42       692
           1       0.23      0.24      0.23       318
           2       0.56      0.37      0.45       821
           3       0.32      0.41      0.36       392
           4       0.12      0.32      0.18       127
           5       0.12      0.23      0.15       147

    accuracy                           0.35      2497
   macro avg       0.31      0.32      0.30      2497
weighted avg       0.42      0.35      0.37      2497



100%|██████████| 25/25 [02:44<00:00,  6.59s/it]

Epoch 24
 train loss: 1.3014210649789908, train acc: 0.47646233439445496, train f1: 0.44577556848526
 val loss: 1.7068631884398733, val acc: 0.35082098841667175, val f1: 0.3015941381454468

              precision    recall  f1-score   support

           0       0.50      0.36      0.42       692
           1       0.24      0.25      0.24       318
           2       0.56      0.38      0.46       821
           3       0.32      0.41      0.36       392
           4       0.13      0.32      0.18       127
           5       0.11      0.22      0.15       147

    accuracy                           0.35      2497
   macro avg       0.31      0.32      0.30      2497
weighted avg       0.42      0.35      0.37      2497



In [82]:
class_weights

array([0.60107869, 1.30838574, 0.50727465, 1.06085331, 3.28128286,
       2.83038549])

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


# Обновляем веса

In [97]:
 weights = np.sqrt(class_weights)

In [98]:
class_weights_tensor2 = torch.tensor(
    weights, dtype=torch.float32
).to(device)

loss_fn_upd = nn.CrossEntropyLoss(weight=class_weights_tensor2)

In [99]:
model4 = CNNModel4()
model4 = model4.to(device)

In [101]:
optimizer = torch.optim.AdamW(
    model4.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

In [102]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)

In [103]:
train_loss_log, train_acc_log, train_f1_log, val_loss_log, val_acc_log, val_f1_log = train(
    model4,
    optimizer,
    20,
    train_loader, 
    val_loader, 
    loss_fn_upd,
    scheduler
)

  0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
  5%|▌     

Epoch 0
 train loss: 1.6911203108536892, train acc: 0.3660857379436493, train f1: 0.19596469402313232
 val loss: 1.6634761231720068, val acc: 0.39967960119247437, val f1: 0.2042248249053955

              precision    recall  f1-score   support

           0       0.37      0.74      0.49       692
           1       0.00      0.00      0.00       318
           2       0.47      0.49      0.48       821
           3       0.33      0.20      0.25       392
           4       0.00      0.00      0.00       127
           5       0.00      0.00      0.00       147

    accuracy                           0.40      2497
   macro avg       0.19      0.24      0.20      2497
weighted avg       0.31      0.40      0.33      2497



 10%|█         | 2/20 [00:13<02:00,  6.72s/it]

Epoch 1
 train loss: 1.649933850344939, train acc: 0.38882210850715637, train f1: 0.21472862362861633
 val loss: 1.6387282571974833, val acc: 0.4064877927303314, val f1: 0.21241402626037598

              precision    recall  f1-score   support

           0       0.39      0.70      0.50       692
           1       0.00      0.00      0.00       318
           2       0.46      0.56      0.50       821
           3       0.36      0.17      0.23       392
           4       0.09      0.03      0.05       127
           5       0.00      0.00      0.00       147

    accuracy                           0.41      2497
   macro avg       0.22      0.24      0.21      2497
weighted avg       0.32      0.41      0.34      2497



 15%|█▌        | 3/20 [00:20<01:53,  6.69s/it]

...

Epoch 17
 train loss: 1.1362184033944056, train acc: 0.5777243375778198, train f1: 0.5358832478523254
 val loss: 1.7940452729061152, val acc: 0.3928714394569397, val f1: 0.2972496747970581

              precision    recall  f1-score   support

           0       0.43      0.49      0.46       692
           1       0.23      0.18      0.20       318
           2       0.51      0.51      0.51       821
           3       0.36      0.34      0.35       392
           4       0.13      0.13      0.13       127
           5       0.13      0.13      0.13       147

    accuracy                           0.39      2497
   macro avg       0.30      0.30      0.30      2497
weighted avg       0.39      0.39      0.39      2497



 95%|█████████▌| 19/20 [02:06<00:06,  6.62s/it]

Epoch 18
 train loss: 1.1061485638985267, train acc: 0.5922476053237915, train f1: 0.552556037902832
 val loss: 1.78908332726758, val acc: 0.3840608596801758, val f1: 0.2921891510486603

              precision    recall  f1-score   support

           0       0.43      0.46      0.45       692
           1       0.23      0.19      0.20       318
           2       0.49      0.49      0.49       821
           3       0.36      0.35      0.36       392
           4       0.13      0.14      0.14       127
           5       0.11      0.12      0.12       147

    accuracy                           0.38      2497
   macro avg       0.29      0.29      0.29      2497
weighted avg       0.38      0.38      0.38      2497



100%|██████████| 20/20 [02:12<00:00,  6.64s/it]

Epoch 19
 train loss: 1.1093627076882582, train acc: 0.5936498641967773, train f1: 0.5544443130493164
 val loss: 1.7772011259558853, val acc: 0.38606327772140503, val f1: 0.2918586730957031

              precision    recall  f1-score   support

           0       0.43      0.47      0.45       692
           1       0.23      0.18      0.20       318
           2       0.49      0.49      0.49       821
           3       0.36      0.36      0.36       392
           4       0.12      0.13      0.12       127
           5       0.12      0.12      0.12       147

    accuracy                           0.39      2497
   macro avg       0.29      0.29      0.29      2497
weighted avg       0.38      0.39      0.38      2497

